In [1]:
# ===== BOOTSTRAP: run first after every Colab start/restart =====
import os, shutil, subprocess
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')

SECRETS = ["GITHUB_USERNAME", "GITHUB_TOKEN", "GIT_EMAIL", "DAGSHUB_USERNAME",
           "DAGSHUB_TOKEN", "KAGGLE_API_TOKEN"]
missing = []
for k in SECRETS:
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        missing.append(k)
print(f"Secrets loaded: {len(SECRETS)-len(missing)}/{len(SECRETS)}  missing: {missing or 'none'}")

PROJECT     = "23L-0951_A2_Ops"
DRIVE       = Path("/content/drive/MyDrive")
PROJECT_DIR = DRIVE / PROJECT
WORK_DIR    = DRIVE / f"{PROJECT}_work"
LOCAL_DATA  = Path("/content/data")
for p in (WORK_DIR, LOCAL_DATA):
    p.mkdir(parents=True, exist_ok=True)
os.environ.update(PROJECT_DIR=str(PROJECT_DIR), WORK_DIR=str(WORK_DIR),
                  LOCAL_DATA=str(LOCAL_DATA))

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out)
    return r.returncode

helper = '!f() { echo "username=${GITHUB_USERNAME}"; echo "password=${GITHUB_TOKEN}"; }; f'
for key, val in [("credential.helper", helper), ("user.name", "Zayan"),
                 ("user.email", os.environ.get("GIT_EMAIL", "")),
                 ("init.defaultBranch", "main")]:
    subprocess.run(["git", "config", "--global", key, val], check=True)

if not (PROJECT_DIR / ".git").exists():
    sh(f'git clone https://github.com/{os.environ["GITHUB_USERNAME"]}/{PROJECT}.git "{PROJECT_DIR}"')
if not (PROJECT_DIR / ".git").exists():
    raise SystemExit("Clone failed: check GITHUB_USERNAME / GITHUB_TOKEN / repo name.")
os.chdir(PROJECT_DIR)
sh("rm -f .git/index.lock")
sh("git pull --ff-only")

if (PROJECT_DIR / "requirements-colab.txt").exists():
    sh("pip install -q -r requirements-colab.txt")

def save_step(message, notebook_name=None):
    if notebook_name:
        src = DRIVE / "Colab Notebooks" / notebook_name
        (PROJECT_DIR / "notebooks").mkdir(exist_ok=True)
        if src.exists():
            shutil.copy(src, PROJECT_DIR / "notebooks" / notebook_name)
        else:
            print("Notebook not found:", src)
    sh("git add -A")
    sh(f'git commit -m "{message}"')
    sh("git push origin main")
    sh("git log --oneline -1")

print("Bootstrap done. Working directory:", os.getcwd())

Mounted at /content/drive
Secrets loaded: 6/6  missing: none
Already up to date.
Bootstrap done. Working directory: /content/drive/MyDrive/23L-0951_A2_Ops


In [2]:
!nvidia-smi

Wed Nov  4 16:29:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   38C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -q mlflow dagshub "dvc[s3]" keras-tuner
!pip install -q -U kaggle

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import sys, platform
import tensorflow as tf, keras, numpy as np, pandas as pd, sklearn, PIL, matplotlib
import mlflow, keras_tuner, dagshub

print("Python      :", sys.version.split()[0], "|", platform.platform())
for name, mod in [("TensorFlow", tf), ("Keras", keras), ("NumPy", np), ("pandas", pd),
                  ("scikit-learn", sklearn), ("Pillow", PIL), ("matplotlib", matplotlib),
                  ("MLflow", mlflow), ("Keras Tuner", keras_tuner), ("dagshub", dagshub)]:
    print(f"{name:12s}:", mod.__version__)
sh("dvc --version");

gpus = tf.config.list_physical_devices("GPU")
print("\nGPUs visible to TensorFlow:", gpus)
print("Built with CUDA:", tf.test.is_built_with_cuda())
if gpus:
    d = tf.config.experimental.get_device_details(gpus[0])
    print("Device name:", d.get("device_name"), "| compute capability:", d.get("compute_capability"))
    with tf.device("/GPU:0"):
        x = tf.random.normal((4096, 4096))
        y = tf.matmul(x, x)
    print("Matmul ran on:", y.device)
else:
    print("NO GPU: check Runtime > Change runtime type")

Python      : 3.13.16 | Linux-6.6.159+-x86_64-with-glibc2.39
TensorFlow  : 2.21.0
Keras       : 3.13.2
NumPy       : 2.4.6
pandas      : 2.2.3
scikit-learn: 1.6.1
Pillow      : 11.3.0
matplotlib  : 3.10.0
MLflow      : 3.18.0
Keras Tuner : 1.4.8
dagshub     : 0.7.3
3.67.1

GPUs visible to TensorFlow: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Built with CUDA: True
Device name: NVIDIA L4 | compute capability: (8, 9)
Matmul ran on: /job:localhost/replica:0/task:0/device:GPU:0


In [5]:
from importlib.metadata import version

key_pkgs = ["tensorflow", "keras", "numpy", "pandas", "scikit-learn", "pillow",
            "matplotlib", "mlflow", "dagshub", "dvc", "dvc-s3", "keras-tuner", "kaggle"]
pins = [f"{p}=={version(p)}" for p in key_pkgs]
(PROJECT_DIR / "requirements-colab.txt").write_text(
    "# Exact versions used for training in Google Colab (A100)\n" + "\n".join(pins) + "\n")
print("\n".join(pins))

env_dir = PROJECT_DIR / "reports" / "env"
env_dir.mkdir(parents=True, exist_ok=True)
sh(f'pip freeze > "{env_dir}/pip-freeze-colab.txt"')
sh(f'nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv > "{env_dir}/gpu.txt"')
print("Saved:", [p.name for p in env_dir.iterdir()])

tensorflow==2.21.0
keras==3.13.2
numpy==2.4.6
pandas==2.2.3
scikit-learn==1.6.1
pillow==11.3.0
matplotlib==3.10.0
mlflow==3.18.0
dagshub==0.7.3
dvc==3.67.1
dvc-s3==3.3.0
keras-tuner==1.4.8
kaggle==2.2.5
Saved: ['pip-freeze-colab.txt', 'gpu.txt']


In [6]:
save_step("Step 2: environment check, pinned requirements, package list", "01_data.ipynb");

[main 0739a2c] Step 2: environment check, pinned requirements, package list
 2 files changed, 15 insertions(+)
 create mode 100644 notebooks/01_data.ipynb
 create mode 100644 requirements-colab.txt
To https://github.com/bhatti-zayan/23L-0951_A2_Ops.git
   22dd27f..0739a2c  main -> main
0739a2c Step 2: environment check, pinned requirements, package list


In [7]:
import hashlib, zipfile

DATASET   = "cashbowman/ai-generated-images-vs-real-images"
ZIP_DIR   = WORK_DIR / "raw_zip"
ZIP_DIR.mkdir(parents=True, exist_ok=True)
RAW_LOCAL = LOCAL_DATA / "raw"

# Download to Drive only if no zip is there yet (reads KAGGLE_API_TOKEN from env)
zips = list(ZIP_DIR.glob("*.zip"))
if not zips:
    sh(f'kaggle datasets download -d {DATASET} -p "{ZIP_DIR}"')
    zips = list(ZIP_DIR.glob("*.zip"))
else:
    print("Zip already on Drive, skipping download.")
if not zips:
    raise SystemExit("Download failed: see the Kaggle message above.")
ZIP_PATH = zips[0]
print("Zip:", ZIP_PATH.name, "| size: %.1f MB" % (ZIP_PATH.stat().st_size / 1e6))

# Unzip to the fast local disk (needed again after every restart)
def ensure_local_raw():
    if RAW_LOCAL.exists() and any(RAW_LOCAL.rglob("*.*")):
        print("Local copy already present:", RAW_LOCAL)
        return
    RAW_LOCAL.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(RAW_LOCAL)
    print("Extracted to", RAW_LOCAL)

ensure_local_raw()

Dataset URL: https://www.kaggle.com/datasets/cashbowman/ai-generated-images-vs-real-images
License(s): Community Data License Agreement - Sharing - Version 1.0


  0%|          | 0.00/476M [00:00<?, ?B/s]
  0%|          | 1.00M/476M [00:01<08:07, 1.02MB/s]
  0%|          | 2.00M/476M [00:01<04:11, 1.98MB/s]
  1%|          | 3.00M/476M [00:01<02:39, 3.11MB/s]
  1%|          | 5.00M/476M [00:01<01:26, 5.73MB/s]
  2%|▏         | 8.00M/476M [00:01<00:49, 9.88MB/s]
  2%|▏         | 11.0M/476M [00:01<00:37, 13.0MB/s]
  3%|▎         | 13.0M/476M [00:01<00:33, 14.6MB/s]
  3%|▎         | 16.0M/476M [00:01<00:28, 17.1MB/s]
  4%|▍         | 18.0M/476M [00:02<00:27, 17.7MB/s]
  4%|▍         | 21.0M/476M [00:02<00:23, 20.1MB/s]
  5%|▌         | 24.0M/476M [00:02<00:23, 20.6MB/s]
  6%|▌         | 27.0M/476M [00:02<00:21, 21.4MB/s]
  6%|▋         | 30.0M/476M [00:02<00:22, 21.2MB/s]
  7%|▋         | 33.0M/476M [00:02<00:22, 21.0MB/s]
  8%|▊         | 36.0M/476M [00:02<00:20, 22.9MB/s]
  8%|▊         

In [8]:
from collections import Counter

total = 0
for d in sorted(p for p in RAW_LOCAL.rglob("*") if p.is_dir()):
    files = [f for f in d.iterdir() if f.is_file()]
    if files:
        total += len(files)
        exts = Counter(f.suffix.lower() or "<none>" for f in files).most_common()
        print(f"{str(d.relative_to(RAW_LOCAL)):45s} {len(files):6d} files  {exts}")
print("\nTotal files:", total)

AiArtData/AiArtData                              539 files  [('.jpg', 385), ('.png', 115), ('.jpeg', 36), ('.webp', 3)]
RealArt/RealArt                                  436 files  [('.jpg', 378), ('.png', 35), ('.jpeg', 21), ('.gif', 1), ('<none>', 1)]

Total files: 975


In [9]:
from datetime import datetime, timezone

h = hashlib.sha256()
with open(ZIP_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)

(PROJECT_DIR / "data" / "SOURCE.md").write_text(f"""# Dataset source
- Kaggle: https://www.kaggle.com/datasets/{DATASET}
- Downloaded: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}
- Zip file: {ZIP_PATH.name} ({ZIP_PATH.stat().st_size} bytes)
- Zip SHA-256: {h.hexdigest()}
- Labels come from folder names (see Step 4 audit).
""")
print((PROJECT_DIR / "data" / "SOURCE.md").read_text())

# Dataset source
- Kaggle: https://www.kaggle.com/datasets/cashbowman/ai-generated-images-vs-real-images
- Downloaded: 2026-11-04 16:36 UTC
- Zip file: ai-generated-images-vs-real-images.zip (499048119 bytes)
- Zip SHA-256: de9243eaa0170bd2bc85963e9141e82158382f950416667eed3d7cdaefce2653
- Labels come from folder names (see Step 4 audit).

